# Banana Detection — YOLOv8 Training

**Dataset:** 837 gambar pisang berlabel (mentah, mengkal, matang, busuk)  
**Model:** YOLOv8n (nano) — ringan, cocok untuk Android  

---

### Checklist sebelum mulai:
- [ ] Runtime sudah diset ke **GPU (T4)**: Runtime → Change runtime type → T4 GPU
- [ ] Folder `banana_dataset_colab` sudah diupload ke Google Drive
- [ ] Struktur folder di Drive:
```
MyDrive/
  banana_dataset_colab/
    train/
      images/  (670 gambar)
      labels/  (670 label)
    val/
      images/  (167 gambar)
      labels/  (167 label)
    dataset.yaml
    classes.txt
```

## 1. Konfigurasi

In [ ]:
# =============================================================
# SESUAIKAN KONFIGURASI DI SINI
# =============================================================

# Nama folder dataset di Google Drive (bukan full path, cukup nama foldernya)
DATASET_FOLDER = 'banana_dataset_colab'

# Model dasar YOLOv8
# yolov8n.pt = paling ringan (recommended untuk Android)
# yolov8s.pt = lebih akurat, lebih berat
BASE_MODEL = 'yolov8n.pt'

# Nama project & run (hasil tersimpan di sini)
PROJECT_NAME = 'banana_detection'
RUN_NAME     = 'v2_yolov8n'

# Hyperparameter
EPOCHS     = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16    # kurangi ke 8 jika RAM GPU habis
PATIENCE   = 25    # early stop jika tidak ada improvement N epoch

# =============================================================

import os
DATASET_PATH = f'/content/drive/MyDrive/{DATASET_FOLDER}'
YAML_PATH    = os.path.join(DATASET_PATH, 'dataset.yaml')

print('Konfigurasi:')
print(f'  Dataset    : {DATASET_PATH}')
print(f'  Base model : {BASE_MODEL}')
print(f'  Epochs     : {EPOCHS}')
print(f'  Image size : {IMAGE_SIZE}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Patience   : {PATIENCE}')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive berhasil di-mount')

## 3. Cek & Fix Dataset

In [ ]:
import os
from collections import defaultdict

print('=== Cek Folder ===')
ok = True
for split in ['train', 'val']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    n_img = len([f for f in os.listdir(img_dir) if not f.startswith('.')]) if os.path.isdir(img_dir) else 0
    n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
    status = 'OK' if n_img == n_lbl and n_img > 0 else 'MASALAH'
    print(f'  [{status}] {split:5s}: {n_img} gambar | {n_lbl} label')
    if n_img != n_lbl:
        ok = False

if not ok:
    print('\nAda ketidaksesuaian jumlah gambar dan label!')
    print('Pastikan prepare_dataset_local.py sudah dijalankan dengan benar.')
else:
    print('\nStruktur folder OK!')

print()
print('=== Distribusi Kelas ===')
class_names = {0:'mentah', 1:'mengkal', 2:'matang', 3:'busuk'}
counts = defaultdict(int)
total_img = 0
for split in ['train', 'val']:
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    if not os.path.isdir(lbl_dir):
        continue
    for fname in os.listdir(lbl_dir):
        if not fname.endswith('.txt'):
            continue
        total_img += 1
        with open(os.path.join(lbl_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1

total_bbox = sum(counts.values())
print(f'  Total gambar : {total_img}')
print(f'  Total bbox   : {total_bbox}')
print()
for cls_id, name in class_names.items():
    count = counts.get(cls_id, 0)
    pct   = count / total_bbox * 100 if total_bbox else 0
    bar   = '=' * int(pct / 2)
    print(f'  {cls_id} {name:8s}: {count:5d} bbox ({pct:5.1f}%) [{bar}]')

print()
print('=== dataset.yaml ===')
with open(YAML_PATH) as f:
    print(f.read())

## 4. Fix dataset.yaml untuk Colab

In [ ]:
# dataset.yaml dari Windows pakai path Windows (D:\...)
# Cell ini otomatis menggantinya ke path Colab yang benar

yaml_content = f"""# Banana Detection Dataset
train: {DATASET_PATH}/train/images
val:   {DATASET_PATH}/val/images

nc: 4

names:
  0: mentah
  1: mengkal
  2: matang
  3: busuk
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print('dataset.yaml sudah diupdate ke path Colab:')
print(yaml_content)

## 5. Install Ultralytics

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()
print('Ultralytics siap!')

## 6. Training

> **Catatan class imbalance:** Dataset ini memiliki distribusi tidak merata  
> (busuk 57% vs mentah 9.7%). Script ini sudah mengaktifkan augmentasi  
> yang kuat untuk membantu model belajar kelas minoritas dengan baik.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data    = YAML_PATH,
    epochs  = EPOCHS,
    imgsz   = IMAGE_SIZE,
    batch   = BATCH_SIZE,
    patience= PATIENCE,
    project = PROJECT_NAME,
    name    = RUN_NAME,
    device  = 0,
    workers = 2,
    cache   = True,

    # Optimizer
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    weight_decay = 0.0005,
    warmup_epochs= 3,

    # Augmentasi — penting untuk atasi class imbalance
    augment   = True,
    fliplr    = 0.5,       # flip horizontal
    flipud    = 0.1,       # flip vertikal
    hsv_h     = 0.015,     # variasi warna (hue)
    hsv_s     = 0.7,       # variasi saturasi
    hsv_v     = 0.4,       # variasi kecerahan
    degrees   = 15,        # rotasi
    translate = 0.1,       # geser
    scale     = 0.5,       # zoom
    shear     = 2.0,       # geser diagonal
    mosaic    = 1.0,       # mosaic augmentation (sangat efektif)
    mixup     = 0.1,       # mixup augmentation
    copy_paste= 0.1,       # copy-paste augmentation

    # Simpan
    save        = True,
    save_period = 10,
    verbose     = True,
)

print()
print('=== TRAINING SELESAI ===')
metrics = results.results_dict
print(f"mAP50      : {metrics.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP50-95   : {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"Precision  : {metrics.get('metrics/precision(B)', 0):.4f}")
print(f"Recall     : {metrics.get('metrics/recall(B)', 0):.4f}")

## 7. Lihat Grafik Hasil Training

In [ ]:
import glob
from IPython.display import Image, display

run_dir = os.path.join(PROJECT_NAME, RUN_NAME)
print(f'Hasil training ada di: {run_dir}')
print()

charts = [
    ('results.png',                   'Loss & Metrics per Epoch'),
    ('confusion_matrix_normalized.png','Confusion Matrix (Normalized)'),
    ('confusion_matrix.png',           'Confusion Matrix (Raw)'),
    ('PR_curve.png',                   'Precision-Recall Curve'),
    ('F1_curve.png',                   'F1 Curve'),
    ('val_batch0_pred.jpg',            'Prediksi Batch Val'),
]

for fname, title in charts:
    full = os.path.join(run_dir, fname)
    if os.path.exists(full):
        print(f'--- {title} ---')
        display(Image(full, width=800))

## 8. Validasi Per Kelas

In [ ]:
best_pt = os.path.join(PROJECT_NAME, RUN_NAME, 'weights', 'best.pt')
model_best = YOLO(best_pt)

val_res = model_best.val(
    data    = YAML_PATH,
    imgsz   = IMAGE_SIZE,
    batch   = BATCH_SIZE,
    device  = 0,
    verbose = True,
)

print()
print('=== Hasil Per Kelas ===')
class_names_list = ['mentah', 'mengkal', 'matang', 'busuk']
for i, name in enumerate(class_names_list):
    try:
        ap50    = float(val_res.box.ap50[i])
        ap5095  = float(val_res.box.ap[i])
        print(f'  {name:8s}: AP@50={ap50:.4f}  AP@50-95={ap5095:.4f}')
    except Exception:
        pass

## 9. Export & Simpan ke Drive

In [ ]:
import shutil

best_pt  = os.path.join(PROJECT_NAME, RUN_NAME, 'weights', 'best.pt')
save_dir = '/content/drive/MyDrive/banana_model'
os.makedirs(save_dir, exist_ok=True)

# Simpan .pt ke Drive
dest_pt = os.path.join(save_dir, 'banana_detection_v2.pt')
shutil.copy2(best_pt, dest_pt)
print(f'Model .pt disimpan : {dest_pt}')
print(f'Ukuran             : {os.path.getsize(dest_pt)/1024/1024:.1f} MB')

# Export TFLite untuk Android
print()
print('Mengekspor ke TFLite untuk Android...')
model_best = YOLO(best_pt)
model_best.export(
    format = 'tflite',
    imgsz  = IMAGE_SIZE,
    int8   = False,
    half   = False,
)

tflite_files = glob.glob(os.path.join(PROJECT_NAME, RUN_NAME, 'weights', '*.tflite'))
if tflite_files:
    dest_tflite = os.path.join(save_dir, 'banana_detection_v2.tflite')
    shutil.copy2(tflite_files[0], dest_tflite)
    print(f'Model .tflite disimpan : {dest_tflite}')
    print(f'Ukuran                 : {os.path.getsize(dest_tflite)/1024/1024:.1f} MB')
else:
    print('File TFLite tidak ditemukan, coba export manual.')

print()
print('=== SELESAI! ===')
print(f'Semua file tersimpan di: {save_dir}')

## Selesai!

File tersimpan di Google Drive folder `banana_model/`:
- **`banana_detection_v2.pt`** — untuk backend Flask (ganti file lama di `trained_models/`)
- **`banana_detection_v2.tflite`** — untuk Android app

### Cara pasang model baru ke backend:
1. Download `banana_detection_v2.pt` dari Drive
2. Upload ke Replit, ganti file di:
   ```
   trained_models/banana_detection_v1/weights/best.pt
   ```
3. Restart server Flask — model baru langsung aktif!

### Update `CLASS_NAMES` di backend jika perlu:
Sesuaikan di `.env` atau `app/config.py`:
```
CLASS_NAMES=Mentah,Mengkal,Matang,Busuk
```